# Can we predict failure within 90 days of an inspection?
Target: `failed_90d`. Plan: inspect the data, check for leakage, build baselines, evaluate on a **time-ordered holdout** (train on earlier inspections, test on the latest), and report honestly.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, RocCurveDisplay, precision_recall_curve
from sklearn.inspection import permutation_importance

df = pd.read_csv('data.csv', parse_dates=['inspected_at']).sort_values('inspected_at').reset_index(drop=True)
print(df.shape)
print(df.isna().sum().sum(), 'missing;', df.duplicated().sum(), 'duplicate rows;', df.asset_tag.nunique(), 'unique assets')
print('Date range:', df.inspected_at.min(), '->', df.inspected_at.max())
print('Base rate:', df.failed_90d.mean().round(4))
df.describe().T

## Leakage check: `parts_replaced`
The dictionary says this is the parts count on the work order *that closed this inspection record*, so it can be recorded after the outcome is known (e.g. repairs following a failure). Let's see.

In [ ]:
df.groupby('parts_replaced').failed_90d.agg(['mean', 'count'])

Every record with 2+ parts replaced failed, and none with 0 failed. That near-perfect separation is almost certainly a consequence of the failure, not a predictor of it, and it would not be available at inspection time. **`parts_replaced` is excluded** from all models (worth confirming with whoever owns the work-order system). The effect of including it is shown at the end for contrast.

## Exploration of legitimate features

In [ ]:
num = ['vibration_rms', 'bearing_temp_c', 'ambient_temp_c', 'meter_hours', 'load_pct']
fig, ax = plt.subplots(1, 5, figsize=(18, 3.2))
for a, col in zip(ax, num):
    for y, l in [(0, 'ok'), (1, 'failed')]:
        a.hist(df.loc[df.failed_90d == y, col], bins=30, alpha=.55, density=True, label=l)
    a.set_title(col)
ax[0].legend(); plt.tight_layout(); plt.show()
print(df.groupby('site').failed_90d.agg(['mean', 'count']))
print(df[num + ['failed_90d']].corr()['failed_90d'].drop('failed_90d').round(3))
print(df.groupby(df.inspected_at.dt.to_period('Q')).failed_90d.agg(['mean', 'count']))

## Evaluation setup
Each asset appears once, so there is no asset-level leakage across rows. Inspections span two years, so I hold out the **latest 25%** by time as the test set and compare models with forward-chaining CV on the training part only.

In [ ]:
feats = num + ['site']
cut = int(len(df) * .75)
tr, te = df.iloc[:cut], df.iloc[cut:]
Xtr, ytr, Xte, yte = tr[feats], tr.failed_90d, te[feats], te.failed_90d
print('train', len(tr), ytr.mean().round(3), '| test', len(te), yte.mean().round(3))

cats = pd.CategoricalDtype(sorted(df.site.unique()))
def gb_frame(X): return X.assign(site=X.site.astype(cats))

pre = ColumnTransformer([('n', StandardScaler(), num), ('c', OneHotEncoder(handle_unknown='ignore'), ['site'])])
models = {
    'baseline (prior)': (DummyClassifier(strategy='prior'), lambda X: X),
    'logistic': (make_pipeline(pre, LogisticRegression(max_iter=1000)), lambda X: X),
    'grad. boosting': (HistGradientBoostingClassifier(max_depth=3, learning_rate=.05, max_iter=200,
                                                      categorical_features='from_dtype', random_state=0), gb_frame),
}
rows = []
for n, (m, prep) in models.items():
    aucs = []
    for a, b in TimeSeriesSplit(5).split(Xtr):
        m.fit(prep(Xtr.iloc[a]), ytr.iloc[a])
        aucs.append(roc_auc_score(ytr.iloc[b], m.predict_proba(prep(Xtr.iloc[b]))[:, 1]))
    rows.append((n, np.mean(aucs), np.std(aucs)))
pd.DataFrame(rows, columns=['model', 'cv_auc', 'sd']).round(3)

In [ ]:
res = {}
for n, (m, prep) in models.items():
    m.fit(prep(Xtr), ytr)
    res[n] = m.predict_proba(prep(Xte))[:, 1]
pd.DataFrame({n: {'ROC AUC': roc_auc_score(yte, p), 'PR AUC (AP)': average_precision_score(yte, p)}
              for n, p in res.items()}).T.round(3)

In [ ]:
rng = np.random.default_rng(0)
p, y = res['logistic'], yte.values
bs = [roc_auc_score(y[i], p[i]) for i in (rng.integers(0, len(y), len(y)) for _ in range(500))]
print('logistic test AUC 95%% CI: %.3f - %.3f' % tuple(np.percentile(bs, [2.5, 97.5])))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for n, p in res.items():
    RocCurveDisplay.from_predictions(yte, p, name=n, ax=ax[0])
    pr, rc, _ = precision_recall_curve(yte, p); ax[1].plot(rc, pr, label=n)
ax[1].axhline(yte.mean(), ls=':', c='gray'); ax[1].set_xlabel('recall'); ax[1].set_ylabel('precision'); ax[1].legend()
plt.show()

In [ ]:
lr = models['logistic'][0]
print(pd.Series(lr[-1].coef_[0], lr[0].get_feature_names_out()).sort_values().round(3))
pi = permutation_importance(lr, Xte, yte, scoring='roc_auc', n_repeats=20, random_state=0)
print(pd.Series(pi.importances_mean, feats).sort_values(ascending=False).round(4))

## Operational view: flagging the riskiest inspections
If maintenance can only follow up on a fraction of inspections, how many failures does the model capture?

In [ ]:
p = res['logistic']; o = np.argsort(-p)
for f in [.05, .1, .2, .3]:
    k = int(len(p) * f); hit = yte.values[o[:k]]
    print(f'top {f:.0%}: precision {hit.mean():.2f} (base {yte.mean():.2f}), recall {hit.sum() / yte.sum():.2f}')

## For contrast: what including `parts_replaced` would do

In [ ]:
f2 = feats + ['parts_replaced']
m = HistGradientBoostingClassifier(categorical_features='from_dtype', random_state=0).fit(gb_frame(tr[f2]), ytr)
print('AUC with leaky feature:', round(roc_auc_score(yte, m.predict_proba(gb_frame(te[f2]))[:, 1]), 3))

## Conclusions (from the run on this data)
- **Signal exists but is modest.** On the time-ordered holdout (latest 25%, base rate 4.3%): gradient boosting ROC AUC ~0.74 (AP ~0.12, ~2.7x base rate); logistic ROC AUC ~0.66 (95% CI roughly 0.59-0.72). Forward-chaining CV AUCs on the training period were ~0.72 (logistic) and ~0.76 (boosting).
- **Drivers:** vibration, load and bearing temperature push risk up; meter hours and ambient temperature point the other way in the linear model. Site carries essentially no signal.
- **`parts_replaced` is leakage** (AUC ~0.997 if used). Do not use it for a real prediction at inspection time.
- **The failure rate drifts strongly over time**: ~27% in 2024 Q2-Q3 vs ~3% in 2025 Q4, while feature means barely move. This hurts the holdout (train 14% vs test 4%) and means probabilities will not be calibrated across periods; use the model for ranking, or recalibrate. It may also reflect right-censoring of labels near the end of the data window, or a process change (e.g. a maintenance programme); worth checking with the data owner.
- **Practical use:** with a 4% base rate, precision is low even when ranking is useful. Treat this as a triage aid, not an automatic failure alarm.
- **Next steps:** per-asset history (trends in vibration/temperature) would likely help but each asset appears only once here; confirm the label cut-off date; tune/calibrate the boosting model.